# PravahX â€” Delft3D Flexible Mesh Builder (Colab)

This notebook compiles Delft3D FM on Google Colab using Intel oneAPI, Conan 2, and Ninja, then exports it to Google Drive.


In [ ]:
%%python
import os

from google.colab import drive

drive.mount("/content/drive")
print("\n\u2713 Google Drive mounted.")

In [ ]:
%%bash
echo "Installing Intel oneAPI toolchain, Conan, Ninja, and Patchelf..."
wget -qO- https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
sudo apt-get update -qq
sudo apt-get install -y -qq intel-oneapi-compiler-fortran intel-oneapi-compiler-dpcpp-cpp-and-cpp-classic intel-oneapi-mpi-devel ninja-build patchelf python3-pip
python3 -m pip install -q "conan>=2.0"


In [ ]:
%%bash
echo "Cloning Deltares/Delft3D repository..."
rm -rf /content/delft3d_src
git clone https://github.com/Deltares/Delft3D.git /content/delft3d_src
cd /content/delft3d_src
git checkout 761dc502e7fe3fac61ecece8c93b94b635f3643b


In [ ]:
%%bash
echo "Initializing Conan external profile and installing dependencies..."
cd /content/delft3d_src
conan profile detect --force
conan config install https://github.com/Deltares/conan-config.git
conan install . -pr:b delft3d_alma8_intel_2024_v3 -pr:h delft3d_alma8_intel_2024_v3 --build=missing


In [ ]:
%%python
print("Patching CMakeLists for UNIX shared library linking (dflowfm-cli, dflowfm_dll)...")
cli_path = "/content/delft3d_src/src/engines_gpl/dflowfm/cli/CMakeLists.txt"
dll_path = "/content/delft3d_src/src/engines_gpl/dflowfm/dll/CMakeLists.txt"


def patch_cmakelists(path):
    if not os.path.exists(path):
        return
    with open(path) as f:
        content = f.read()
    if "if(UNIX)" in content and "target_include_directories" not in content:
        content += (
            "\n# patched: force link interface\nif(UNIX)\n"
            "    target_include_directories(${TARGET_NAME} PRIVATE ${mpi_module_path})\n"
            "    target_link_options(${TARGET_NAME} PRIVATE ${openmp_flag})\nendif()\n"
        )
        with open(path, "w") as f:
            f.write(content)


patch_cmakelists(cli_path)
patch_cmakelists(dll_path)

In [ ]:
%%bash
echo "Running CMake configuration with Ninja and Intel compilers..."
source /opt/intel/oneapi/setvars.sh
cd /content/delft3d_src
cmake -S src/cmake -B build_ninja -G Ninja \
    -DCONFIGURATION_TYPE=dflowfm \
    -DCMAKE_TOOLCHAIN_FILE=build/generators/conan_toolchain.cmake \
    -DCMAKE_C_COMPILER=icx \
    -DCMAKE_CXX_COMPILER=icpx \
    -DCMAKE_Fortran_COMPILER=ifx \
    -Dmpi_include_path=/opt/intel/oneapi/mpi/latest/include \
    -DCMAKE_Fortran_STANDARD_LIBRARIES="-lifcore -limf -liomp5 -lmpi" \
    -DCMAKE_CXX_STANDARD_LIBRARIES="-limf -liomp5 -lmpi" \
    -DCMAKE_INSTALL_PREFIX=/content/delft3d_bin


In [ ]:
%%bash
echo "Building with Ninja..."
source /opt/intel/oneapi/setvars.sh
cd /content/delft3d_src
cmake --build build_ninja
echo "Applying patchelf..."
find build_ninja -type f -executable -name "*dflowfm*" -exec patchelf --set-rpath "\$ORIGIN/../lib" {} \;
echo "Installing..."
cmake --install build_ninja


In [ ]:
%%bash
echo "Packaging to Google Drive..."
tar -czvf /content/delft3dfm_linux_x86_64.tar.gz -C /content delft3d_bin
sha256sum /content/delft3dfm_linux_x86_64.tar.gz > /content/delft3dfm_linux_x86_64.tar.gz.sha256
cp /content/delft3dfm_linux_x86_64.tar.gz* /content/drive/MyDrive/PravahX/
echo "\u2713 Build, package, and Drive export complete."
